### 0. Configuración del Entorno
Ejecuta la siguiente celda para asegurarte de tener instaladas todas las librerías necesarias para el análisis.

In [ ]:
%pip install pandas numpy scipy matplotlib seaborn openpyxl

# Etapa 2: Parámetros e Inferencias Estadísticas del Desarrollo Humano
En esta sección se desarrollan formalmente los cálculos correspondientes a la estimación puntual, la estimación por intervalos de confianza y la estimación de máxima verosimilitud (MLE). El estudio se basa en una muestra significativa de 10,000 registros de edades, con el fin de inferir el comportamiento estadístico de la población y validar las métricas obtenidas durante la Etapa 1 del proyecto. Se busca determinar con precisión cómo se distribuyen las edades a través de las distintas etapas biológicas y sociales del ser humano.

In [ ]:
# Se importan las librerías con las que se realizará el procesamiento
import pandas as pd
import numpy as np
from scipy import stats
from scipy.optimize import minimize

# Se establecen los parámetros base para el estudio técnico
rutaExcel = 'Datos Etapa 2.xlsx'
muPoblacional = 40
sigmaPoblacional = 30
limitesEtapas = [-1, 12, 18, 25, 60, 100]
etiquetasEtapas = ['Niñez', 'Adolescencia', 'Juventud', 'Adultez', 'Vejez']

### 1. Carga, limpieza y estructuración de la muestra
Se procede con la carga de los datos crudos desde el archivo Excel. Dado que la información se encuentra organizada en una cuadrícula de 100x100, se realiza una operación de aplanado para consolidar los 10,000 registros en una estructura lineal que facilite su manipulación estadística. Posteriormente, se separa la muestra total en dos subgrupos equitativos (5,000 registros para hombres y 5,000 para mujeres) siguiendo las especificaciones del diseño experimental. Finalmente, se clasifica cada registro dentro de su etapa de desarrollo correspondiente (Niñez, Adolescencia, Juventud, Adultez o Vejez) para permitir un análisis segmentado y comparativo.

In [ ]:
# Se carga el archivo desde la hoja específica de la muestra
dfCrudo = pd.read_excel(rutaExcel, sheet_name='Muestra de edades', header=None, skiprows=1)

# Se transforma la matriz de 100x100 en un vector unidimensional de 10,000 elementos
muestraTotal = dfCrudo.values.flatten()
muestraTotal = muestraTotal[~np.isnan(muestraTotal)]

print(f"Total de registros procesados con éxito: {len(muestraTotal)}")

# Se segmenta la muestra para realizar comparaciones por sexo (50% hombres, 50% mujeres)
hombresMuestra = muestraTotal[:5000]
mujeresMuestra = muestraTotal[5000:10000]

dfHombres = pd.DataFrame({'Edad': hombresMuestra, 'Sexo': 'Hombre'})
dfMujeres = pd.DataFrame({'Edad': mujeresMuestra, 'Sexo': 'Mujer'})
dfTotal = pd.concat([dfHombres, dfMujeres]).reset_index(drop=True)

# Se asigna la etiqueta de desarrollo humano basada en los rangos definidos
dfTotal['Etapa'] = pd.cut(dfTotal['Edad'], bins=limitesEtapas, labels=etiquetasEtapas)
dfTotal.head()

### 2. Estimación puntual de la media poblacional por sexo
En esta etapa se calcula la media muestral para cada uno de los sexos identificados en la base de datos. Estos estimadores puntuales representan la mejor aproximación del centro de los datos. Adicionalmente, se realiza una comparación directa contra los parámetros poblacionales históricos establecidos en la Etapa 1 ($\mu = 40$), calculando el error absoluto para evaluar la representatividad y el posible sesgo de la muestra recolectada.

In [ ]:
# Se calculan promedios y varianzas por sexo
estadisticasPorSexo = dfTotal.groupby('Sexo')['Edad'].agg(['mean', 'std', 'count', 'var']).reset_index()
estadisticasPorSexo.columns = ['Sexo', 'Media Muestral', 'Desv. Estándar', 'n', 'Varianza Muestral']

# Se compara contra la media poblacional
estadisticasPorSexo['Error Absoluto (vs MU=40)'] = abs(estadisticasPorSexo['Media Muestral'] - muPoblacional)

print("Resultados de estimación por sexo:")
display(estadisticasPorSexo)

### 3. Estimación por el método de Máxima Verosimilitud (MLE)
Se emplea el método de Máxima Verosimilitud (Maximum Likelihood Estimation) para estimar los parámetros óptimos de la distribución normal ($\mu$ y $\sigma$) que maximizan la probabilidad de haber observado los datos actuales. Para lograr esto, se minimiza la función de log-verosimilitud negativa mediante algoritmos de optimización numérica (L-BFGS-B). Este procedimiento se repite de manera iterativa para cada etapa del desarrollo humano, permitiendo capturar las diferencias estructurales en el promedio de edad y la variabilidad de cada subgrupo poblacional.

In [ ]:
# Se define la función de log-verosimilitud negativa para una distribución normal
def logVerosimilitud(parametros, datos):
    mu, sigma = parametros
    if sigma <= 0: return 1e10 # Penalización para desviaciones no válidas
    return -np.sum(stats.norm.logpdf(datos, loc=mu, scale=sigma))

resultadosMLE = []
for etapa in etiquetasEtapas:
    # Se filtran los datos correspondientes a la etapa actual de la iteración
    datosEtapa = dfTotal[dfTotal['Etapa'] == etapa]['Edad'].values
    if len(datosEtapa) == 0: continue
    
    # Se inicializan las semillas para la optimización y se ejecutan las restricciones
    parametrosIniciales = [np.mean(datosEtapa), np.std(datosEtapa)]
    solucion = minimize(logVerosimilitud, parametrosIniciales, args=(datosEtapa,), method='L-BFGS-B', 
                       bounds=[(0, 100), (0.1, 50)])
    
    # Se extraen y almacenan los resultados óptimos encontrados por el algoritmo
    muMLE, sigmaMLE = solucion.x
    resultadosMLE.append({
        'Etapa': etapa,
        'Media (MLE)': muMLE,
        'Sigma (MLE)': sigmaMLE,
        'Densidad Max (PDF)': stats.norm.pdf(muMLE, loc=muMLE, scale=sigmaMLE),
        'n': len(datosEtapa)
    })

dfMLE = pd.DataFrame(resultadosMLE)
print("Resultados de los parámetros óptimos encontrados mediante MLE por etapa:")
display(dfMLE)

### 4. Estimación por Intervalos de Confianza (Nivel de confianza: 95%)
Se calculan los intervalos dentro de los cuales se espera encontrar los parámetros reales de la población con una probabilidad del 95%. Para la media, se utiliza la distribución normal dada la gran cantidad de datos, mientras que para la varianza se hace uso de la distribución Chi-cuadrado ($\chi^2$). Estos rangos proporcionan una medida de la incertidumbre estadística y permiten confirmar que la variabilidad observada en la muestra es consistente con los supuestos teóricos.

In [ ]:
# Se establecen los intervalos del 95% para la media muestral y la varianza por etapa
dfMLE['IC_Inferior_Media'] = dfMLE['Media (MLE)'] - 1.96 * (sigmaPoblacional / np.sqrt(dfMLE['n']))
dfMLE['IC_Superior_Media'] = dfMLE['Media (MLE)'] + 1.96 * (sigmaPoblacional / np.sqrt(dfMLE['n']))

alfa = 0.05
dfMLE['IC_Inferior_Var'] = ((dfMLE['n'] - 1) * (dfMLE['Sigma (MLE)']**2)) / stats.chi2.ppf(1 - alfa/2, dfMLE['n'] - 1)
dfMLE['IC_Superior_Var'] = ((dfMLE['n'] - 1) * (dfMLE['Sigma (MLE)']**2)) / stats.chi2.ppf(alfa/2, dfMLE['n'] - 1)

display(dfMLE[['Etapa', 'Media (MLE)', 'IC_Inferior_Media', 'IC_Superior_Media', 'IC_Inferior_Var', 'IC_Superior_Var']])

## V. Conclusiones
Basado en los resultados obtenidos:
- La media muestral para ambos sexos se encuentra cerca del parámetro poblacional histórico (40), con un error absoluto mínimo.
- El modelo de Máxima Verosimilitud permitió ajustar los parámetros específicos para cada etapa de desarrollo.
- Los intervalos de confianza confirman la validez estadística de la muestra extraída.